# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` data science library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a single object

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Record sets (@id): {metadata.recordSet}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get the available record sets
record_set_ids = metadata.recordSet
if not record_set_ids:
    print("No record sets found in the metadata. Check the schema definition.")
else:
    for rs_id in record_set_ids:
        print(f"Record set @id: {rs_id}")
        # Get field IDs for the record set
        fields = dataset.fields(record_set=rs_id)
        print(f"  Fields in record set {rs_id}:")
        for field in fields:
            print(f"   - {field['@id']} ({field['name']})")

## 3. Data Extraction
Load data from all available record sets into DataFrames for further analysis.
Record sets and field `@id`s are used, as shown above.

In [ ]:
# Extract and preview data from each record set
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set {rs_id} columns: {df.columns.tolist()}")
        print(f"Head of {rs_id}:")
        display(df.head())
    else:
        print(f"No records found for record set {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. Replace the following example IDs with actual field `@id`s identified previously.

In [ ]:
# Example: EDA on main survey record set
# Replace the below IDs with real record set and field IDs

# Choose a record set containing survey responses
if dataframes:
    # Arbitrarily pick the first record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Show all available columns for reference
    print("Available columns in the DataFrame:")
    print(df.columns.tolist())

    # Example numeric field: GAD-7 score (replace with correct @id if needed)
    # For demonstration, let's assume there's a column 'gad_7_score'
    numeric_field = 'gad_7_score' if 'gad_7_score' in df.columns else df.columns[-1]  # fallback
    threshold = 10

    if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Example group field: village (replace with correct @id if needed)
        group_field = 'village' if 'village' in df.columns else df.columns[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print(f"Group field '{group_field}' not found in DataFrame columns.")
    else:
        print(f"No numeric field available for EDA in {record_set_id} record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize summary statistics and relationships among fields in the dataset. Example: distribution of GAD-7 scores, PHQ-9 scores, or relationships between scores and demographic variables.

In [ ]:
# Visualization example
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Example: score distribution histogram
    score_field = 'gad_7_score' if 'gad_7_score' in df.columns else df.columns[-1]
    if score_field in df.columns and pd.api.types.is_numeric_dtype(df[score_field]):
        plt.figure(figsize=(8,4))
        sns.histplot(df[score_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {score_field} in Record Set {record_set_id}")
        plt.xlabel(score_field)
        plt.ylabel("Frequency")
        plt.show()

    # Example: boxplot of scores by group
    group_field = 'village' if 'village' in df.columns else df.columns[0]
    if group_field in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=score_field, data=df)
        plt.title(f"Boxplot of {score_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed survey responses capturing mental health indicators and demographics for residents of Kilifi County, Kenya.
- Using `mlcroissant`, we loaded and explored record sets and fields identified by `@id`, enabling reproducible analyses.
- Exploratory analysis and visualizations revealed distributions of mental health scores and their relationships to demographic groups.
- The dataset can further be used for public health research and strategy development targeting community mental health.